In [ ]:
import warnings
warnings.filterwarnings('ignore')
import gymnasium as gym
import numpy as np

from gym_env import SalesNegotiationEnv


class FlattenObservationWrapper(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        # We redefine the observation space as a Box (continuous vector)
        low = np.zeros(env.observation_space.shape, dtype=np.float32)
        high = np.array([max(s.nvec) for s in [env.observation_space]], dtype=np.float32).flatten()
        
        self.observation_space = gym.spaces.Box(
            low=0, 
            high=100, # A safe upper bound for your round/count/action indices
            shape=env.observation_space.shape, 
            dtype=np.float32
        )

    def observation(self, observation):
        # Convert the MultiDiscrete array to a float32 array
        return np.array(observation, dtype=np.float32)


env_base = SalesNegotiationEnv()
env = FlattenObservationWrapper(env_base)

In [22]:
''' Format of observation:
[current round, A/B/C resolved, Incentive Used] + [Action History for all max_rounds] + [Topic History] 
index
current round: 0
A/B/C resolved: 1,2,3
Incentive Used: 4
Action History: 5 - 35
Topic History: 36 - 66
'''

obj, latent = env.reset()
print(f"Initial Observation: {obj},\n\nDetermination and Patience: {latent}")

def unpack_observation(obs):
    state = {}
    state['current_round'] = obs[0]
    state['resolved'] = {
        'A': obs[1] == 1,
        'B': obs[2] == 1,
        'C': obs[3] == 1
    }

    state['incentive_used'] = obs[4] == 1
    state['action_history'] = obs[5:36]
    state['topic_history'] = obs[36:67]

    return state

state = unpack_observation(obj)
print(state)

print(f"length of state['action_history']: {len(state['action_history'])}")
print(f"length of state['topic_history']: {len(state['topic_history'])}")

episode_reward = 0

Initial Observation: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 3. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.],

Determination and Patience: {'latent_D': 0.8075006254906422, 'latent_P': 0.726528834363164}
{'current_round': np.float32(0.0), 'resolved': {'A': np.False_, 'B': np.False_, 'C': np.False_}, 'incentive_used': np.False_, 'action_history': array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
      dtype=float32), 'topic_history': array([3., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
      dtype=float32)}
length of state['action_history']: 31
length of state['topic_history']: 31


In [26]:
obj, reward, terminated, truncated, info = env.step(0)
episode_reward += reward
print(f"Observation: {unpack_observation(obj)},\nInstant Reward: {reward}, \nEpisode Reward: {episode_reward}, \nTerminated: {terminated}, \nTruncated: {truncated}, \n Info: {info}")

Observation: {'current_round': np.float32(4.0), 'resolved': {'A': np.True_, 'B': np.False_, 'C': np.True_}, 'incentive_used': np.False_, 'action_history': array([1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
      dtype=float32), 'topic_history': array([3., 3., 1., 2., 2., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
      dtype=float32)},
Instant Reward: 9.8, 
Episode Reward: 9.200000000000001, 
Terminated: True, 
Truncated: False, 
 Info: {}
